# GSA Analysis: Precipitation and LAI Control on Evaporative Fluxes

## Task Overview
- **Objective**: Quantify how strongly precipitation and LAI control different evaporative fluxes
- **Models**: CLASSIC and LPJ-GUESS
- **Method**: Global Sensitivity Analysis (GSA)
- **Time Period**: 1981-2020 (40 years)

## Step 1: Data Preprocessing

### 1.1 Import Libraries and Setup

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Define data paths
data_dir = Path('data')
classic_dir = data_dir / 'CLASSIC'
lpj_dir = data_dir / 'LPJ-GUESS'

print("Libraries imported successfully!")
print(f"Data directory: {data_dir.absolute()}")

Libraries imported successfully!
Data directory: /Users/mimi/Documents/Code/Github/GSA-work/explore/data


### 1.2 Load and Explore NetCDF Files

In [2]:
# Define variables to load
evap_vars = ['evapotrans', 'tran', 'evspsblveg', 'evspsblsoi']

# Function to load all variables for a model
def load_model_data(model_dir, model_name):
    """Load all NetCDF files for a given model"""
    data = {}
    
    # Load precipitation (shared between models)
    precip_file = data_dir / 'precipitation_annual.nc'
    print(f"\nLoading precipitation from: {precip_file.name}")
    data['precipitation'] = xr.open_dataset(precip_file)
    
    # Load LAI
    lai_file = model_dir / f'{model_name}_lai.nc'
    print(f"Loading LAI from: {lai_file.name}")
    data['lai'] = xr.open_dataset(lai_file)
    
    # Load evaporative fluxes
    for var in evap_vars:
        file_path = model_dir / f'{model_name}_{var}.nc'
        print(f"Loading {var} from: {file_path.name}")
        data[var] = xr.open_dataset(file_path)
    
    return data

# Load CLASSIC data
print("="*60)
print("Loading CLASSIC Model Data")
print("="*60)
classic_data = load_model_data(classic_dir, 'CLASSIC')

# Load LPJ-GUESS data
print("\n" + "="*60)
print("Loading LPJ-GUESS Model Data")
print("="*60)
lpj_data = load_model_data(lpj_dir, 'LPJ_GUESS')

print("\n" + "="*60)
print("Data loading completed!")
print("="*60)

Loading CLASSIC Model Data

Loading precipitation from: precipitation_annual.nc
Loading LAI from: CLASSIC_lai.nc
Loading evapotrans from: CLASSIC_evapotrans.nc
Loading tran from: CLASSIC_tran.nc
Loading evspsblveg from: CLASSIC_evspsblveg.nc
Loading evspsblsoi from: CLASSIC_evspsblsoi.nc

Loading LPJ-GUESS Model Data

Loading precipitation from: precipitation_annual.nc
Loading LAI from: LPJ_GUESS_lai.nc
Loading evapotrans from: LPJ_GUESS_evapotrans.nc
Loading tran from: LPJ_GUESS_tran.nc
Loading evspsblveg from: LPJ_GUESS_evspsblveg.nc
Loading evspsblsoi from: LPJ_GUESS_evspsblsoi.nc

Data loading completed!


### 1.3 Explore Data Structure

In [3]:
# Examine the structure of precipitation data
print("PRECIPITATION DATA STRUCTURE:")
print("="*60)
print(classic_data['precipitation'])
print("\nVariable names:", list(classic_data['precipitation'].data_vars))
print("Dimensions:", dict(classic_data['precipitation'].dims))
print("Coordinates:", list(classic_data['precipitation'].coords))

# Examine CLASSIC LAI structure
print("\n\nCLASSIC LAI DATA STRUCTURE:")
print("="*60)
print(classic_data['lai'])
print("\nVariable names:", list(classic_data['lai'].data_vars))

# Examine CLASSIC evapotrans structure
print("\n\nCLASSIC EVAPOTRANS DATA STRUCTURE:")
print("="*60)
print(classic_data['evapotrans'])
print("\nVariable names:", list(classic_data['evapotrans'].data_vars))

PRECIPITATION DATA STRUCTURE:
<xarray.Dataset> Size: 1GB
Dimensions:        (latitude: 1500, longitude: 3600, time: 34)
Coordinates:
  * latitude       (latitude) float64 12kB 89.95 89.85 89.75 ... -59.85 -59.95
  * longitude      (longitude) float64 29kB -179.9 -179.8 -179.8 ... 179.8 179.9
  * time           (time) int32 136B 1991 1992 1993 1994 ... 2021 2022 2023 2024
Data variables:
    precipitation  (latitude, longitude, time) float64 1GB ...

Variable names: ['precipitation']
Dimensions: {'latitude': 1500, 'longitude': 3600, 'time': 34}
Coordinates: ['latitude', 'longitude', 'time']


CLASSIC LAI DATA STRUCTURE:
<xarray.Dataset> Size: 166MB
Dimensions:  (time: 40, lon: 720, lat: 360)
Coordinates:
  * time     (time) uint32 160B 1981 1982 1983 1984 1985 ... 2017 2018 2019 2020
  * lon      (lon) float32 3kB -179.8 -179.2 -178.8 -178.2 ... 178.8 179.2 179.8
  * lat      (lat) float32 1kB -89.75 -89.25 -88.75 -88.25 ... 88.75 89.25 89.75
Data variables:
    data_S0  (time, lat, lon

### 1.4 Extract and Align Variables

**Problem identified**: 
- Precipitation: (1500, 3600, 34) - years 1991-2024, lat **-59.95° to 89.95°**, 0.1° resolution
- Model data: (40, 360, 720) - years 1981-2020, lat **-89.75° to 89.75°**, 0.5° resolution
- **Issue**: Precipitation does NOT cover southern high latitudes (Antarctica)

**Solution**:
1. Extract variables from datasets
2. Filter model grid to precipitation's latitude range (**-59.75° to 89.75°**)
   - Removes 60 latitude points (Antarctic region below -60°)
   - Keeps 300 latitude points
3. Align precipitation to filtered model grid (select matching points)
4. Subset all variables to time overlap (1991-2020, 30 years)
5. **Final shape: (30, 300, 720)** for all variables

In [4]:
def extract_variables(data_dict, model_name):
    """
    Extract DataArrays from datasets and organize them
    """
    extracted = {}

    # Get the actual variable names from the datasets
    for key, dataset in data_dict.items():
        var_names = list(dataset.data_vars)
        if len(var_names) > 0:
            # Take the first (and usually only) data variable
            # var_name = var_names[0]
            if var_names[0] == 'precipitation':
                var_name = var_names[0]
            else:
                var_name = var_names[1]
            extracted[key] = dataset[var_name]
            print(f"{key:15s} -> variable: {var_name:20s} | shape: {extracted[key].shape}")
        else:
            print(f"Warning: No data variables found in {key}")
    
    return extracted

# Extract variables for both models
print("CLASSIC Model Variables (Before Alignment):")
print("-" * 60)
classic_vars = extract_variables(classic_data, 'CLASSIC')

print("\n\nLPJ-GUESS Model Variables (Before Alignment):")
print("-" * 60)
lpj_vars = extract_variables(lpj_data, 'LPJ-GUESS')

CLASSIC Model Variables (Before Alignment):
------------------------------------------------------------
precipitation   -> variable: precipitation        | shape: (1500, 3600, 34)
lai             -> variable: data_S1              | shape: (40, 360, 720)
evapotrans      -> variable: data_S1              | shape: (40, 360, 720)
tran            -> variable: data_S1              | shape: (40, 360, 720)
evspsblveg      -> variable: data_S1              | shape: (40, 360, 720)
evspsblsoi      -> variable: data_S1              | shape: (40, 360, 720)


LPJ-GUESS Model Variables (Before Alignment):
------------------------------------------------------------
precipitation   -> variable: precipitation        | shape: (1500, 3600, 34)
lai             -> variable: data_S1              | shape: (40, 360, 720)
evapotrans      -> variable: data_S1              | shape: (40, 360, 720)
tran            -> variable: data_S1              | shape: (40, 360, 720)
evspsblveg      -> variable: data_S1      

### 1.4.5 Align Precipitation Data to Model Grid

**Key constraints identified**: 
- Precipitation latitude range: **-59.95° to 89.95°** (does not cover Antarctica)
- Model latitude range: **-89.75° to 89.75°** (full global coverage)
- **Solution**: Restrict model grid to precipitation's latitude coverage (**-59.75° to 89.75°**)

**Alignment steps**:
1. Filter model latitudes to match precipitation coverage
2. Select time overlap: 1991-2020 (30 years)  
3. Select spatial points using exact coordinate matching (no interpolation needed)

In [5]:
def align_precipitation_to_model(precip_data, model_data, model_name):
    """
    Align precipitation data to model grid (time and space)
    
    Steps:
    1. Select time overlap (1991-2020)
    2. Filter model grid to precipitation's latitude range (-59.75 to 89.75)
    3. Select spatial points matching filtered model grid
    4. Reorder dimensions to match model (time, lat, lon)
    """
    print(f"\nAligning precipitation to {model_name} grid...")
    print("="*60)
    
    # Get model coordinates
    model_time = model_data['lai'].time.values
    model_lat = model_data['lai'].lat.values
    model_lon = model_data['lai'].lon.values
    
    print(f"Original model grid:")
    print(f"  Time: {model_time.min()} - {model_time.max()} ({len(model_time)} years)")
    print(f"  Lat:  {model_lat.min():.2f} to {model_lat.max():.2f} ({len(model_lat)} points)")
    print(f"  Lon:  {model_lon.min():.2f} to {model_lon.max():.2f} ({len(model_lon)} points)")
    
    # Get precipitation coordinates
    precip = model_data['precipitation']
    precip_lat_min = precip.latitude.values.min()
    precip_lat_max = precip.latitude.values.max()
    
    print(f"\nPrecipitation data:")
    print(f"  Time: {precip.time.values.min()} - {precip.time.values.max()} ({len(precip.time)} years)")
    print(f"  Lat:  {precip_lat_max:.2f} to {precip_lat_min:.2f} ({len(precip.latitude)} points)")
    print(f"  Lon:  {precip.longitude.values.min():.2f} to {precip.longitude.values.max():.2f} ({len(precip.longitude)} points)")
    
    # Filter model latitudes to precipitation's range
    # Precipitation covers: -59.95 to 89.95
    # We need model points within this range: -59.75 to 89.75
    lat_mask = (model_lat >= precip_lat_min) & (model_lat <= precip_lat_max)
    model_lat_filtered = model_lat[lat_mask]
    
    print(f"\nFiltered model grid to precipitation's latitude range:")
    print(f"  Lat:  {model_lat_filtered.min():.2f} to {model_lat_filtered.max():.2f} ({len(model_lat_filtered)} points)")
    print(f"  Removed: {len(model_lat) - len(model_lat_filtered)} latitude points (south of {precip_lat_min:.2f}°)")
    
    # Step 1: Select time overlap (1991-2020)
    time_overlap = [t for t in model_time if t in precip.time.values]
    print(f"\nTime overlap: {min(time_overlap)} - {max(time_overlap)} ({len(time_overlap)} years)")
    
    # Step 2: Select spatial points and time
    # Use sel() to select exact matching coordinates
    precip_aligned = precip.sel(
        time=time_overlap,
        latitude=model_lat_filtered,
        longitude=model_lon
    )
    
    # Rename coordinates to match model naming convention
    precip_aligned = precip_aligned.rename({'latitude': 'lat', 'longitude': 'lon'})
    
    # Reorder dimensions to (time, lat, lon) to match model
    precip_aligned = precip_aligned.transpose('time', 'lat', 'lon')
    
    print(f"\n✓ Aligned precipitation shape: {precip_aligned.shape}")
    print(f"  Expected shape: ({len(time_overlap)}, {len(model_lat_filtered)}, {len(model_lon)})")
    
    return precip_aligned, time_overlap, model_lat_filtered

# Align precipitation for CLASSIC
classic_precip_aligned, classic_time, classic_lat_filtered = align_precipitation_to_model(
    classic_vars['precipitation'], 
    classic_vars, 
    'CLASSIC'
)

print("\n" + "="*60)

# Align precipitation for LPJ-GUESS  
lpj_precip_aligned, lpj_time, lpj_lat_filtered = align_precipitation_to_model(
    lpj_vars['precipitation'],
    lpj_vars,
    'LPJ-GUESS'
)

print("\n" + "="*60)
print("Precipitation alignment completed!")
print("="*60)


Aligning precipitation to CLASSIC grid...
Original model grid:
  Time: 1981 - 2020 (40 years)
  Lat:  -89.75 to 89.75 (360 points)
  Lon:  -179.75 to 179.75 (720 points)

Precipitation data:
  Time: 1991 - 2024 (34 years)
  Lat:  89.95 to -59.95 (1500 points)
  Lon:  -179.95 to 179.95 (3600 points)

Filtered model grid to precipitation's latitude range:
  Lat:  -59.75 to 89.75 (300 points)
  Removed: 60 latitude points (south of -59.95°)

Time overlap: 1991 - 2020 (30 years)

✓ Aligned precipitation shape: (30, 300, 720)
  Expected shape: (30, 300, 720)


Aligning precipitation to LPJ-GUESS grid...
Original model grid:
  Time: 1981 - 2020 (40 years)
  Lat:  -89.75 to 89.75 (360 points)
  Lon:  -179.75 to 179.75 (720 points)

Precipitation data:
  Time: 1991 - 2024 (34 years)
  Lat:  89.95 to -59.95 (1500 points)
  Lon:  -179.95 to 179.95 (3600 points)

Filtered model grid to precipitation's latitude range:
  Lat:  -59.75 to 89.75 (300 points)
  Removed: 60 latitude points (south of -5

### 1.4.6 Subset Model Variables to Time Overlap

In [6]:
def subset_model_variables(vars_dict, time_overlap, lat_filtered, model_name):
    """
    Subset model variables (LAI and evaporative fluxes) to:
    1. Time overlap period (1991-2020)
    2. Filtered latitude range (matching precipitation coverage)
    """
    print(f"\nSubsetting {model_name} variables...")
    print("-" * 60)
    
    vars_aligned = {}
    
    # Add aligned precipitation
    if model_name == 'CLASSIC':
        vars_aligned['precipitation'] = classic_precip_aligned
    else:
        vars_aligned['precipitation'] = lpj_precip_aligned
    
    # Subset other variables to time overlap AND latitude range
    for var_name, data_array in vars_dict.items():
        if var_name == 'precipitation':
            continue  # Already handled
        
        # Select time overlap and latitude range
        data_subset = data_array.sel(time=time_overlap, lat=lat_filtered)
        vars_aligned[var_name] = data_subset
        
        print(f"  {var_name:15s}: {data_array.shape} -> {data_subset.shape}")
    
    return vars_aligned

# Subset CLASSIC variables
classic_vars_aligned = subset_model_variables(
    classic_vars, 
    classic_time, 
    classic_lat_filtered, 
    'CLASSIC'
)

print()

# Subset LPJ-GUESS variables
lpj_vars_aligned = subset_model_variables(
    lpj_vars, 
    lpj_time, 
    lpj_lat_filtered, 
    'LPJ-GUESS'
)

print("\n" + "="*60)
print("All variables aligned!")
print("="*60)
print(f"\nFinal dimensions:")
print(f"  Time: {len(classic_time)} years (1991-2020)")
print(f"  Lat:  {len(classic_lat_filtered)} points ({classic_lat_filtered.min():.2f}° to {classic_lat_filtered.max():.2f}°)")
print(f"  Lon:  720 points")
print(f"\nTotal samples per model: {len(classic_time) * len(classic_lat_filtered) * 720:,}")


Subsetting CLASSIC variables...
------------------------------------------------------------
  lai            : (40, 360, 720) -> (30, 300, 720)
  evapotrans     : (40, 360, 720) -> (30, 300, 720)
  tran           : (40, 360, 720) -> (30, 300, 720)
  evspsblveg     : (40, 360, 720) -> (30, 300, 720)
  evspsblsoi     : (40, 360, 720) -> (30, 300, 720)


Subsetting LPJ-GUESS variables...
------------------------------------------------------------
  lai            : (40, 360, 720) -> (30, 300, 720)
  evapotrans     : (40, 360, 720) -> (30, 300, 720)
  tran           : (40, 360, 720) -> (30, 300, 720)
  evspsblveg     : (40, 360, 720) -> (30, 300, 720)
  evspsblsoi     : (40, 360, 720) -> (30, 300, 720)

All variables aligned!

Final dimensions:
  Time: 30 years (1991-2020)
  Lat:  300 points (-59.75° to 89.75°)
  Lon:  720 points

Total samples per model: 6,480,000


### 1.5 Check Data Quality (Missing Values, Dimensions)

In [7]:
def check_data_quality(vars_dict, model_name):
    """
    Check for missing values, NaNs, and basic statistics
    """
    print(f"\n{'='*60}")
    print(f"Data Quality Check: {model_name}")
    print(f"{'='*60}\n")
    
    quality_report = {}
    
    for var_name, data_array in vars_dict.items():
        print(f"\n{var_name.upper()}")
        print("-" * 40)
        
        # Check dimensions and shape
        print(f"  Shape: {data_array.shape}")
        print(f"  Dims:  {data_array.dims}")
        
        # Check for NaN values
        n_total = data_array.size
        n_nan = np.isnan(data_array.values).sum()
        pct_nan = (n_nan / n_total) * 100
        
        print(f"  Total values: {n_total:,}")
        print(f"  NaN values:   {n_nan:,} ({pct_nan:.2f}%)")
        
        # Basic statistics (excluding NaN)
        valid_data = data_array.values[~np.isnan(data_array.values)]
        if len(valid_data) > 0:
            print(f"  Min:    {valid_data.min():.4f}")
            print(f"  Max:    {valid_data.max():.4f}")
            print(f"  Mean:   {valid_data.mean():.4f}")
            print(f"  Median: {np.median(valid_data):.4f}")
            print(f"  Std:    {valid_data.std():.4f}")
        
        quality_report[var_name] = {
            'shape': data_array.shape,
            'n_total': n_total,
            'n_nan': n_nan,
            'pct_nan': pct_nan,
            'min': valid_data.min() if len(valid_data) > 0 else np.nan,
            'max': valid_data.max() if len(valid_data) > 0 else np.nan,
            'mean': valid_data.mean() if len(valid_data) > 0 else np.nan,
            'std': valid_data.std() if len(valid_data) > 0 else np.nan
        }
    
    return quality_report

# Check data quality for both models AFTER alignment
classic_quality = check_data_quality(classic_vars_aligned, 'CLASSIC (Aligned)')
lpj_quality = check_data_quality(lpj_vars_aligned, 'LPJ-GUESS (Aligned)')


Data Quality Check: CLASSIC (Aligned)


PRECIPITATION
----------------------------------------
  Shape: (30, 300, 720)
  Dims:  ('time', 'lat', 'lon')
  Total values: 6,480,000
  NaN values:   3,712,297 (57.29%)
  Min:    0.0000
  Max:    14118.3750
  Mean:   451.2380
  Median: 230.8125
  Std:    652.4859

LAI
----------------------------------------
  Shape: (30, 300, 720)
  Dims:  ('time', 'lat', 'lon')
  Total values: 6,480,000
  NaN values:   4,623,750 (71.35%)
  Min:    0.0000
  Max:    9.2171
  Mean:   1.3742
  Median: 0.5466
  Std:    1.7672

EVAPOTRANS
----------------------------------------
  Shape: (30, 300, 720)
  Dims:  ('time', 'lat', 'lon')
  Total values: 6,480,000
  NaN values:   4,623,750 (71.35%)
  Min:    -193.7225
  Max:    2001.7495
  Mean:   394.5540
  Median: 282.9929
  Std:    340.5737

TRAN
----------------------------------------
  Shape: (30, 300, 720)
  Dims:  ('time', 'lat', 'lon')
  Total values: 6,480,000
  NaN values:   4,623,750 (71.35%)
  Min:    0.0

### 1.6 Data Flattening and Integration

Convert multi-dimensional arrays (time × lat × lon) into flat DataFrames suitable for GSA analysis.

In [8]:
def flatten_and_integrate(vars_dict, model_name):
    """
    Flatten multidimensional arrays and create a unified DataFrame
    
    All variables should now have the same shape: (time, lat, lon)
    
    Returns:
        DataFrame with columns: [time, lat, lon, precipitation, lai, evapotrans, tran, evspsblveg, evspsblsoi]
    """
    print(f"\nFlattening data for {model_name}...")
    print("="*60)
    
    # Verify all arrays have the same shape
    shapes = {name: arr.shape for name, arr in vars_dict.items()}
    print("Variable shapes:")
    for name, shape in shapes.items():
        print(f"  {name:15s}: {shape}")
    
    # Check if all shapes are identical
    unique_shapes = set(shapes.values())
    if len(unique_shapes) != 1:
        raise ValueError(f"Not all arrays have the same shape: {shapes}")
    
    print("\n✓ All variables have matching dimensions!")
    
    # Stack all dimensions into a single dimension
    flattened_data = {}
    
    for var_name, data_array in vars_dict.items():
        print(f"  Flattening {var_name}... ", end='')
        
        # Stack all dimensions and convert to 1D array
        stacked = data_array.stack(samples=data_array.dims)
        flattened_data[var_name] = stacked.values
        
        print(f"✓ (shape: {stacked.shape[0]:,})")
    
    # Get coordinates from the first variable
    first_var = list(vars_dict.values())[0]
    stacked_coords = first_var.stack(samples=first_var.dims)
    
    # Extract coordinate values
    coords_dict = {}
    for coord_name in ['time', 'lat', 'lon']:
        if coord_name in stacked_coords.coords:
            coords_dict[coord_name] = stacked_coords[coord_name].values
    
    # Create DataFrame with coordinates and data
    df = pd.DataFrame(coords_dict)
    
    # Add variable data
    for var_name, values in flattened_data.items():
        df[var_name] = values
    
    print(f"\n✓ Created DataFrame with shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    
    return df

# Flatten data for both models using ALIGNED variables
print("\n" + "="*60)
print("FLATTENING ALIGNED DATA")
print("="*60)

classic_df = flatten_and_integrate(classic_vars_aligned, 'CLASSIC')
lpj_df = flatten_and_integrate(lpj_vars_aligned, 'LPJ-GUESS')

print("\n" + "="*60)
print("Data Flattening Completed!")
print("="*60)


FLATTENING ALIGNED DATA

Flattening data for CLASSIC...
Variable shapes:
  precipitation  : (30, 300, 720)
  lai            : (30, 300, 720)
  evapotrans     : (30, 300, 720)
  tran           : (30, 300, 720)
  evspsblveg     : (30, 300, 720)
  evspsblsoi     : (30, 300, 720)

✓ All variables have matching dimensions!
  Flattening precipitation... ✓ (shape: 6,480,000)
  Flattening lai... ✓ (shape: 6,480,000)
  Flattening evapotrans... ✓ (shape: 6,480,000)
  Flattening tran... ✓ (shape: 6,480,000)
  Flattening evspsblveg... ✓ (shape: 6,480,000)
  Flattening evspsblsoi... ✓ (shape: 6,480,000)

✓ Created DataFrame with shape: (6480000, 9)
  Columns: ['time', 'lat', 'lon', 'precipitation', 'lai', 'evapotrans', 'tran', 'evspsblveg', 'evspsblsoi']

Flattening data for LPJ-GUESS...
Variable shapes:
  precipitation  : (30, 300, 720)
  lai            : (30, 300, 720)
  evapotrans     : (30, 300, 720)
  tran           : (30, 300, 720)
  evspsblveg     : (30, 300, 720)
  evspsblsoi     : (30, 30

### 1.7 Remove Missing Values and Summary

In [9]:
# Remove rows with any NaN values
print("Removing rows with NaN values...")
print("="*60)

print(f"\nCLASSIC:")
print(f"  Before: {len(classic_df):,} rows")
classic_df_clean = classic_df.dropna()
print(f"  After:  {len(classic_df_clean):,} rows")
print(f"  Removed: {len(classic_df) - len(classic_df_clean):,} rows ({((len(classic_df) - len(classic_df_clean))/len(classic_df)*100):.2f}%)")

print(f"\nLPJ-GUESS:")
print(f"  Before: {len(lpj_df):,} rows")
lpj_df_clean = lpj_df.dropna()
print(f"  After:  {len(lpj_df_clean):,} rows")
print(f"  Removed: {len(lpj_df) - len(lpj_df_clean):,} rows ({((len(lpj_df) - len(lpj_df_clean))/len(lpj_df)*100):.2f}%)")

# Display summary of cleaned data
print("\n\n" + "="*60)
print("PREPROCESSED DATA SUMMARY")
print("="*60)

print("\nCLASSIC Model:")
print(classic_df_clean.describe())

Removing rows with NaN values...

CLASSIC:
  Before: 6,480,000 rows
  After:  1,824,800 rows
  Removed: 4,655,200 rows (71.84%)

LPJ-GUESS:
  Before: 6,480,000 rows
  After:  1,751,623 rows
  Removed: 4,728,377 rows (72.97%)


PREPROCESSED DATA SUMMARY

CLASSIC Model:
               time           lat           lon  precipitation           lai  \
count  1.824800e+06  1.824800e+06  1.824800e+06   1.824800e+06  1.824800e+06   
mean   2.005501e+03  3.264545e+01  1.801680e+01   6.821964e+02  1.381802e+00   
std    8.655321e+00  3.192433e+01  8.470028e+01   6.960183e+02  1.770905e+00   
min    1.991000e+03 -5.525000e+01 -1.797500e+02   0.000000e+00  0.000000e+00   
25%    1.998000e+03  9.750000e+00 -6.175000e+01   2.375625e+02  5.700213e-02   
50%    2.006000e+03  3.975000e+01  2.775000e+01   4.818125e+02  5.560588e-01   
75%    2.013000e+03  5.925000e+01  9.025000e+01   8.811875e+02  2.032432e+00   
max    2.020000e+03  8.325000e+01  1.787500e+02   1.411838e+04  9.217102e+00   

         e

In [10]:
print("\nLPJ-GUESS Model:")
print(lpj_df_clean.describe())

# Show first few rows
print("\n\nSample Data (first 5 rows):")
print("\nCLASSIC:")
print(classic_df_clean.head())

print("\nLPJ-GUESS:")
print(lpj_df_clean.head())


LPJ-GUESS Model:
               time           lat           lon  precipitation           lai  \
count  1.751623e+06  1.751623e+06  1.751623e+06   1.751623e+06  1.751623e+06   
mean   2.005494e+03  3.079221e+01  2.066719e+01   7.071543e+02  1.743178e+00   
std    8.655425e+00  3.135465e+01  8.555481e+01   6.963595e+02  1.631385e+00   
min    1.991000e+03 -5.525000e+01 -1.797500e+02   0.000000e+00  0.000000e+00   
25%    1.998000e+03  8.250000e+00 -6.275000e+01   2.636250e+02  4.109885e-01   
50%    2.005000e+03  3.775000e+01  3.075000e+01   5.016250e+02  1.278248e+00   
75%    2.013000e+03  5.725000e+01  9.375000e+01   9.097500e+02  2.659388e+00   
max    2.020000e+03  8.325000e+01  1.797500e+02   1.411838e+04  1.045420e+01   

         evapotrans          tran    evspsblveg    evspsblsoi  
count  1.751623e+06  1.751623e+06  1.751623e+06  1.751623e+06  
mean   4.631392e+02  3.426285e+02  3.723161e+01  8.327906e+01  
std    3.987791e+02  3.000419e+02  6.320874e+01  8.442080e+01  
min  

### 1.8 Save Preprocessed Data

In [11]:
# Save preprocessed data to CSV files for easy access
output_dir = Path('data/preprocessed')
output_dir.mkdir(exist_ok=True)

print("Saving preprocessed data...")
print("="*60)

# 1. 保存完整的时间序列数据（原始功能）
classic_output = output_dir / 'classic_preprocessed.csv'
lpj_output = output_dir / 'lpj_guess_preprocessed.csv'

classic_df_clean.to_csv(classic_output, index=False)
print(f"✓ CLASSIC完整数据已保存到: {classic_output}")
print(f"  样本数: {len(classic_df_clean):,}")
print(f"  文件大小: {classic_output.stat().st_size / (1024*1024):.2f} MB")

lpj_df_clean.to_csv(lpj_output, index=False)
print(f"\n✓ LPJ-GUESS完整数据已保存到: {lpj_output}")
print(f"  样本数: {len(lpj_df_clean):,}")
print(f"  文件大小: {lpj_output.stat().st_size / (1024*1024):.2f} MB")

# 2. 计算并保存30年平均数据（新功能）
print("\n" + "="*60)
print("计算30年平均数据...")
print("="*60)

# 定义需要平均的变量（除了time之外的所有数值变量）
value_vars = ['precipitation', 'lai', 'evapotrans', 'tran', 'evspsblveg', 'evspsblsoi']

# CLASSIC: 按经纬度分组，计算30年平均值
classic_mean = classic_df_clean.groupby(['lat', 'lon'])[value_vars].mean().reset_index()
classic_mean_output = output_dir / 'classic_preprocessed_30yr_mean.csv'
classic_mean.to_csv(classic_mean_output, index=False)
print(f"\n✓ CLASSIC 30年平均数据已保存到: {classic_mean_output}")
print(f"  样本数: {len(classic_mean):,} (每个空间网格点的30年平均)")
print(f"  文件大小: {classic_mean_output.stat().st_size / (1024*1024):.2f} MB")

# LPJ-GUESS: 按经纬度分组，计算30年平均值
lpj_mean = lpj_df_clean.groupby(['lat', 'lon'])[value_vars].mean().reset_index()
lpj_mean_output = output_dir / 'lpj_guess_preprocessed_30yr_mean.csv'
lpj_mean.to_csv(lpj_mean_output, index=False)
print(f"\n✓ LPJ-GUESS 30年平均数据已保存到: {lpj_mean_output}")
print(f"  样本数: {len(lpj_mean):,} (每个空间网格点的30年平均)")
print(f"  文件大小: {lpj_mean_output.stat().st_size / (1024*1024):.2f} MB")

# 显示30年平均数据的统计信息
print("\n" + "="*60)
print("30年平均数据统计:")
print("="*60)
print("\nCLASSIC模型 - 30年平均值统计:")
print(classic_mean[value_vars].describe())

print("\nLPJ-GUESS模型 - 30年平均值统计:")
print(lpj_mean[value_vars].describe())

print("\n" + "="*60)
print("STEP 1 COMPLETED: Data Preprocessing Finished!")
print("="*60)
print("\n已生成的文件:")
print("\n时间序列数据 (1991-2020, 30年):")
print(f"  1. {classic_output.name}")
print(f"     - 样本数: {len(classic_df_clean):,} (300 lat × 720 lon × 30 years)")
print(f"     - 变量: {len(classic_df_clean.columns)} columns")
print(f"  2. {lpj_output.name}")
print(f"     - 样本数: {len(lpj_df_clean):,}")
print(f"     - 变量: {len(lpj_df_clean.columns)} columns")

print("\n30年平均数据 (空间分布):")
print(f"  3. {classic_mean_output.name}")
print(f"     - 样本数: {len(classic_mean):,} (空间网格点)")
print(f"     - 变量: lat, lon + {len(value_vars)} 平均值")
print(f"  4. {lpj_mean_output.name}")
print(f"     - 样本数: {len(lpj_mean):,} (空间网格点)")
print(f"     - 变量: lat, lon + {len(value_vars)} 平均值")

print(f"\n空间覆盖范围: -59.75° to 89.75° latitude (300 points × 720 longitude)")
print(f"\n💡 使用建议:")
print(f"  • 时序分析/GSA → 使用时间序列数据 (*_preprocessed.csv)")
print(f"  • 空间模式分析 → 使用30年平均数据 (*_30yr_mean.csv)")
print(f"\nReady for analysis!")

Saving preprocessed data...
✓ CLASSIC完整数据已保存到: data/preprocessed/classic_preprocessed.csv
  样本数: 1,824,800
  文件大小: 131.14 MB

✓ LPJ-GUESS完整数据已保存到: data/preprocessed/lpj_guess_preprocessed.csv
  样本数: 1,751,623
  文件大小: 126.79 MB

计算30年平均数据...

✓ CLASSIC 30年平均数据已保存到: data/preprocessed/classic_preprocessed_30yr_mean.csv
  样本数: 61,826 (每个空间网格点的30年平均)
  文件大小: 4.49 MB

✓ LPJ-GUESS 30年平均数据已保存到: data/preprocessed/lpj_guess_preprocessed_30yr_mean.csv
  样本数: 59,070 (每个空间网格点的30年平均)
  文件大小: 4.36 MB

30年平均数据统计:

CLASSIC模型 - 30年平均值统计:
       precipitation           lai    evapotrans          tran    evspsblveg  \
count   61826.000000  61826.000000  61826.000000  61826.000000  61826.000000   
mean      671.940718      1.372964    394.197906    126.825706     98.718430   
std       675.844651      1.761307    337.725006    165.118225    146.398117   
min         0.000000      0.000000   -170.497498      0.000000     -1.093194   
25%       243.603581      0.057067    155.374264      5.679019      5.4228